In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# ======================
# 1. 数据集
# ======================
transform = transforms.Compose([
    transforms.ToTensor(),  # 转为张量，范围 [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # 标准化（MNIST 均值和方差）
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

# ======================
# 2. 定义 CNN 模型
# ======================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# ======================
# 3. 训练与测试
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 训练
for epoch in range(1, 6):  # 训练 5 个 epoch
    model.train()
    total_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Train Loss: {total_loss/len(train_loader):.4f}")

    # 测试
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
    print(f"Test Accuracy: {100. * correct / len(test_dataset):.2f}%")


Epoch 1, Train Loss: 0.1232
Test Accuracy: 98.36%
Epoch 2, Train Loss: 0.0379
Test Accuracy: 98.42%
Epoch 3, Train Loss: 0.0235
Test Accuracy: 98.62%
Epoch 4, Train Loss: 0.0156
Test Accuracy: 98.96%
Epoch 5, Train Loss: 0.0109
Test Accuracy: 98.84%


In [7]:
#导出 CNN 到 ONNX

model.eval()

data_iter = iter(test_loader)
images, labels = next(data_iter)

# 取第一个样本
first_image = images[0].unsqueeze(0)
first_label = labels[0]

print(f"图像形状: {first_image.shape}")
print(f"标签: {first_label}")

dummy_input = first_image.to('cuda')
input_name = ['input']
output_name = ['output']
export_file_name = "cnn_onnx_module.onnx"
torch.onnx.export(model,dummy_input,
                  export_file_name,
                  export_params=True,
                  opset_version=14,
                  do_constant_folding=True,
                  input_names=input_name,
                  output_names=output_name
                  )


图像形状: torch.Size([1, 1, 28, 28])
标签: 7


C:\Users\27427\AppData\Local\Temp\ipykernel_18876\1987114036.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(model,dummy_input,
